## Install PySpark and create a session

In [751]:
!pip install pyspark==4.1.2

In [752]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrameReader

#create a spark session
spark=SparkSession.builder.appName("AirQuality").getOrCreate()
spark

## Read in data to pyspark dataframe

In [753]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [754]:
#define folder path
path='/content/drive/MyDrive/AirQualityModel/SiteData'

#import datetime
from datetime import datetime,date,time

#Read a single CSV to check inferred schema
df=spark.read.csv('/content/drive/MyDrive/AirQualityModel/SiteData/AQ_Site1.csv',header=True,inferSchema=True)

In [755]:
#Check the top 5 rows to see that data has read in as expected
df.show(5)

+----------+--------+------------------------------------------+----------------+-----------------------+-------+-------------------+-------+--------------------+---------+---------+----------------+---------------+-------+
|      Date|    Time|PM2.5 particulate matter (Hourly measured)|         Status3|Modelled Wind Direction|Status5|Modelled Wind Speed|Status7|           Site Name| Latitude|Longitude|       Site Type|Local Authority|   Zone|
+----------+--------+------------------------------------------+----------------+-----------------------+-------+-------------------+-------+--------------------+---------+---------+----------------+---------------+-------+
|01/01/2025|01:00:00|                                     7.453|V ugm-3 (Ref.eq)|                  231.4|  N deg|               10.9| N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Background|      Hertsmere|Eastern|
|01/01/2025|02:00:00|                                     4.151|V ugm-3 (Ref.eq)|                  231.9

In [756]:
#Check the schema
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- PM2.5 particulate matter (Hourly measured): string (nullable = true)
 |-- Status3: string (nullable = true)
 |-- Modelled Wind Direction: string (nullable = true)
 |-- Status5: string (nullable = true)
 |-- Modelled Wind Speed: string (nullable = true)
 |-- Status7: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Site Type: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



#The inferred schema has mis-defined some fields as strings, when they should be dates, times floats and doubles

Define a new schema that better matches the expected data

In [757]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType,DoubleType,TimestampType,DateType,TimeType

#define the schema
schema = StructType([
    StructField("Date", DateType(), nullable=True),
    StructField("Time", TimestampType(), nullable=True),
    StructField("ParticulateReading", DoubleType(), nullable=True),
    StructField("ParticulateStatus", StringType(),nullable=True),
    StructField("WindDirection",FloatType(), nullable=True),
    StructField("WindDirection Status",StringType(),nullable=True),
    StructField("WindSpeed",FloatType(),nullable=True),
    StructField("windSpeedStatus",StringType(),nullable=True),
    StructField("SiteName",StringType(),nullable=True),
    StructField("Latitude",DoubleType(),nullable=True),
    StructField("Longitude",DoubleType(),nullable=True),
    StructField("SiteType",StringType(),nullable=True),
    StructField("LocalAuthority",StringType(),nullable=True),
    StructField("Zone",StringType(),nullable=True)
    ])

#Read a single CSV to confirm that new schema has been applied
df2=spark.read.csv('/content/drive/MyDrive/AirQualityModel/SiteData/AQ_Site1.csv',header=True,schema=schema)
df2.printSchema()



root
 |-- Date: date (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- ParticulateReading: double (nullable = true)
 |-- ParticulateStatus: string (nullable = true)
 |-- WindDirection: float (nullable = true)
 |-- WindDirection Status: string (nullable = true)
 |-- WindSpeed: float (nullable = true)
 |-- windSpeedStatus: string (nullable = true)
 |-- SiteName: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- SiteType: string (nullable = true)
 |-- LocalAuthority: string (nullable = true)
 |-- Zone: string (nullable = true)



In [758]:
df2.show(5)

+----+-------------------+------------------+-----------------+-------------+--------------------+---------+---------------+--------------------+---------+---------+----------------+--------------+-------+
|Date|               Time|ParticulateReading|ParticulateStatus|WindDirection|WindDirection Status|WindSpeed|windSpeedStatus|            SiteName| Latitude|Longitude|        SiteType|LocalAuthority|   Zone|
+----+-------------------+------------------+-----------------+-------------+--------------------+---------+---------------+--------------------+---------+---------+----------------+--------------+-------+
|NULL|2026-07-17 01:00:00|             7.453| V ugm-3 (Ref.eq)|        231.4|               N deg|     10.9|         N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Background|     Hertsmere|Eastern|
|NULL|2026-07-17 02:00:00|             4.151| V ugm-3 (Ref.eq)|        231.9|               N deg|     11.3|         N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Backg

# Having confirmed the new schema is being applied on a single CSV, we now read all the CSVs together into a single dataframe

However, we can see that the data has returned as NULL, as it is not in the default format - when loading in the full dataset, we will use the dataFormat option to define the date format used

In [759]:
#read all csv files in folder to single dataframe
df=(spark.read
    .format('csv')
    .option('header',True)
    .option('dateFormat','dd/MM/yyyy')
    .option('timeFormat',"HH:mm:ss")
    .schema(schema)
    .load(path))
#Check the schema
df.printSchema()


root
 |-- Date: date (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- ParticulateReading: double (nullable = true)
 |-- ParticulateStatus: string (nullable = true)
 |-- WindDirection: float (nullable = true)
 |-- WindDirection Status: string (nullable = true)
 |-- WindSpeed: float (nullable = true)
 |-- windSpeedStatus: string (nullable = true)
 |-- SiteName: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- SiteType: string (nullable = true)
 |-- LocalAuthority: string (nullable = true)
 |-- Zone: string (nullable = true)



Defining the time field as a timestamp has caused an arbitary date to be added; As the data is hourly measurements, we will use the hour() function to add a column containing an integer representing the hour

In [760]:
#add required functions
import pyspark.sql.functions as sf
from pyspark.sql import Column
#Add "hour" column
df=df.withColumn('Hour',sf.hour(df.Time))
#re-cast Time to string format by removing the date element
df=df.withColumn('Time',sf.date_format(sf.col("Time"), "HH:mm:ss"))
#Check output
df.show(5)
df.printSchema()

+----------+--------+------------------+-----------------+-------------+--------------------+---------+---------------+--------------------+---------+---------+----------------+--------------+-------+----+
|      Date|    Time|ParticulateReading|ParticulateStatus|WindDirection|WindDirection Status|WindSpeed|windSpeedStatus|            SiteName| Latitude|Longitude|        SiteType|LocalAuthority|   Zone|Hour|
+----------+--------+------------------+-----------------+-------------+--------------------+---------+---------------+--------------------+---------+---------+----------------+--------------+-------+----+
|2025-01-01|01:00:00|            12.618| V ugm-3 (Ref.eq)|        236.5|               N deg|     13.7|         N ms-1|Peterborough Gart...|52.594012|-0.248945|Urban Background|  Peterborough|Eastern|   1|
|2025-01-01|02:00:00|             3.325| V ugm-3 (Ref.eq)|        236.0|               N deg|     14.2|         N ms-1|Peterborough Gart...|52.594012|-0.248945|Urban Background

# Initial Data Exploration and Cleansing


In [761]:
#Start with using the describe function on the core columns (non-status) to get a picture of the data
df.select("Date","ParticulateReading","WindDirection","WindSpeed","Hour").describe().show()

+-------+------------------+-----------------+------------------+-----------------+
|summary|ParticulateReading|    WindDirection|         WindSpeed|             Hour|
+-------+------------------+-----------------+------------------+-----------------+
|  count|             94254|            94512|             94512|           100740|
|   mean| 8.635067296878514| 190.751974356321| 4.924885728524972|             12.0|
| stddev| 7.937105028580018|96.91355478251096|2.5305540503679733|6.633282503576391|
|    min|              -4.0|              0.0|               0.0|                1|
|    max|           179.647|            360.0|              21.7|               23|
+-------+------------------+-----------------+------------------+-----------------+



There is a spread of values across the target variable (particulate reading); these will be plotted to look in more detail.

We will also examine the data for indications of correlation between the target variable and the others

In [762]:
#We also need to examine the categorical fields, specifically site type
df.groupBy(df.SiteType).count().show()

+----------------+-----+
|        SiteType|count|
+----------------+-----+
|            NULL|   48|
|   Urban Traffic|26280|
|Rural Background|35040|
|Urban Background|43800|
+----------------+-----+



We can see that there are 3 main site types; reasonaly distributed across all three, but with a slight skew towards 'Urban background'.  There is a small amount of null values; given the small number, these rows can be removed from the dataset

Comparing the total count by SiteType minus nulls (105,120) and the counts by numerical fields above, we can see that there is a reasonable amount of missing data (~10%).

In [763]:
#lets examine the data by site name
df.groupBy(df.SiteName).count().show()


+--------------------+-----+
|            SiteName|count|
+--------------------+-----+
|Borehamwood Meado...| 8760|
|                NULL|   48|
|Stanford-le-Hope ...| 8760|
|Peterborough Gart...| 8760|
| Norwich Lakenfields| 8760|
|     Southend-on-Sea| 8760|
|          Wicken Fen| 8760|
|            Thurrock| 8760|
|              Sibton| 8760|
| Luton A505 Roadside| 8760|
|           Weybourne| 8760|
|      Sandy Roadside| 8760|
|            St Osyth| 8760|
+--------------------+-----+



There are the same number of null rows as with SiteType; these are likely the same rows so removing these records is probably the best way to handle the missing data in this case.

In [764]:
#we look at the numerical fields by site name and site type to see how the missing data is distributed.
import pandas as pd
import pyspark.pandas as ps
described = ps.DataFrame(df.select("SiteName","ParticulateReading","WindDirection","WindSpeed","Hour"))
described.groupby('SiteName').describe().sort_index()

ParticulateReading                                                             WindDirection                                                                              WindSpeed                                                       Hour                                            
                                       count       mean       std    min    25%     50%     75%      max         count        mean         std  min         25%         50%         75%         max     count      mean       std  min  25%  50%  75%        max   count  mean       std  min  25%   50%   75%   max
SiteName                                                                                                                                                                                                                                                                                                            
Borehamwood Meadow Park               8647.0   8.213246  7.348208  0.283  3.774   5.755   9.811   66.510        8544.0  182.690836  100.799205  0.1   89.500000  206.600006  255.000000  359.700012    8544.0  3.971524  1.808608  0.0  2.7  3.6  5.0  12.700000  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Luton A505 Roadside                   8604.0   9.000980  7.624132  0.307  4.269   6.604  10.991   78.656        8544.0  192.195353   95.244523  0.1  117.400002  212.699997  262.799988  359.600006    8544.0  4.769089  2.262578  0.0  3.1  4.4  6.1  15.200000  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Norwich Lakenfields                   8563.0   7.934342  6.989364  0.519  3.703   5.542   9.434   62.571        8544.0  194.295330   96.587515  0.0  118.199997  214.500000  264.399994  360.000000    8544.0  4.468340  2.180971  0.0  2.9  4.1  5.7  15.500000  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Peterborough Garton End               8750.0   8.322098  7.997052  0.330  3.538   5.566   9.835  179.647        8544.0  193.991292   95.821781  0.0  118.400002  216.699997  260.899994  359.899994    8544.0  4.483193  2.176456  0.0  2.9  4.1  5.7  14.800000  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Sandy Roadside                         505.0  13.617822  9.465606 -2.000  6.000  12.000  20.000   50.000         528.0  227.123674   71.853953  0.2  198.899994  231.800003  271.299988  359.200012     528.0  3.854924  2.546243  0.5  2.1  3.2  4.7  14.100000  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Sibton                                8702.0   7.570853  7.367717  0.377  3.325   5.142   8.585   63.562        8544.0  197.074333   97.812392  0.1  112.900002  218.899994  269.700012  360.000000    8544.0  4.696067  2.241588  0.0  3.1  4.3  5.9  16.900000  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Southend-on-Sea                       8541.0   8.689885  8.037322  0.307  4.009   5.873  10.307   70.920        8544.0  187.292439   97.871523  0.1   99.300003  209.199997  260.299988  360.000000    8544.0  5.956929  2.771241  0.1  3.9  5.7  7.7  21.400000  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
St Osyth                              8754.0   8.405682  7.702916  0.354  3.939   5.873   9.646   64.623        8544.0  189.187640   97.550909  0.0   99.199997  208.699997  262.299988  359.899994    8544.0  6.533017  3.067655  0.3  4.3  6.1  8.5  21.700001  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Stanford-le-Hope Roadside             8049.0  11.271587  9.153840 -4.000  6.000   9.000  13.000  107.000        8544.0  188.974087   98.396317  0.0  100.500000  209.500000  265.200012  359.899994    8544.0  4.258088  2.154374  0.0  2.7  3.8  5.4  16.799999  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Thurrock                              8751.0   9.036760  8.541364  0.377  4.057   6.038  10.660   79.151        8544.0  188.958310   96.924938  0.0  103.800003  208.399994  264.000000  360.000000    8544.0  4.088577  1.987831  0.0  2.7  3.7  5.2  14.300000  8395.0  12.0  6.633645  1.0  6.0  12.0  18.0  23.0
Weybourne

In [765]:
#provide a count by columns to demonstrate where missing vaules are
described.groupby('SiteName').count().sort_index()

,ParticulateReading,WindDirection,WindSpeed,Hour
SiteName,,,,
Borehamwood Meadow Park,8647,8544,8544,8395
Luton A505 Roadside,8604,8544,8544,8395
Norwich Lakenfields,8563,8544,8544,8395
Peterborough Garton End,8750,8544,8544,8395
Sandy Roadside,505,528,528,8395
Sibton,8702,8544,8544,8395
Southend-on-Sea,8541,8544,8544,8395
St Osyth,8754,8544,8544,8395
Stanford-le-Hope Roadside,8049,8544,8544,8395


In [766]:
#Look at site type by site name
df.groupBy('SiteName','SiteType').count().show()

+--------------------+----------------+-----+
|            SiteName|        SiteType|count|
+--------------------+----------------+-----+
|                NULL|            NULL|   48|
| Norwich Lakenfields|Urban Background| 8760|
|Peterborough Gart...|Urban Background| 8760|
|Borehamwood Meado...|Urban Background| 8760|
|          Wicken Fen|Rural Background| 8760|
|Stanford-le-Hope ...|   Urban Traffic| 8760|
|     Southend-on-Sea|Urban Background| 8760|
|      Sandy Roadside|   Urban Traffic| 8760|
|            St Osyth|Rural Background| 8760|
|           Weybourne|Rural Background| 8760|
|            Thurrock|Urban Background| 8760|
|              Sibton|Rural Background| 8760|
| Luton A505 Roadside|   Urban Traffic| 8760|
+--------------------+----------------+-----+



In [767]:
#Average values by site type
df.groupBy('SiteType').avg().show()

+----------------+-----------------------+------------------+------------------+-----------------+--------------------+---------+
|        SiteType|avg(ParticulateReading)|avg(WindDirection)|    avg(WindSpeed)|    avg(Latitude)|      avg(Longitude)|avg(Hour)|
+----------------+-----------------------+------------------+------------------+-----------------+--------------------+---------+
|            NULL|                   NULL|              NULL|              NULL|             NULL|                NULL|     NULL|
|   Urban Traffic|       10.2020300734351|191.67989326653864| 4.493846501688135|51.84762566667087|-0.10762266666670735|     12.0|
|Rural Background|      8.088941762203019|191.90659527678181| 5.561031133439854| 52.3303159999867|  0.9813602499998204|     12.0|
|Urban Background|      8.440790159992552| 189.4456413951772|4.5937125464460085|51.97826000002274|  0.3558981999998343|     12.0|
+----------------+-----------------------+------------------+------------------+----------

In [768]:
#Average values by Site Name
df.groupBy('SiteName').avg().show()

+--------------------+-----------------------+------------------+------------------+------------------+--------------------+---------+
|            SiteName|avg(ParticulateReading)|avg(WindDirection)|    avg(WindSpeed)|     avg(Latitude)|      avg(Longitude)|avg(Hour)|
+--------------------+-----------------------+------------------+------------------+------------------+--------------------+---------+
|Borehamwood Meado...|       8.21324563432402|182.69083569121585|3.9715238750854693| 51.66122900000589|-0.27054999999999835|     12.0|
|                NULL|                   NULL|              NULL|              NULL|              NULL|                NULL|     NULL|
|Stanford-le-Hope ...|     11.271586532488508| 188.9740870716469|  4.25808754541531|51.518166999989596|  0.4395479999999297|     12.0|
|Peterborough Gart...|      8.322097828571435|193.99129212309245| 4.483192883916757| 52.59401200000346|-0.24894500000002223|     12.0|
| Norwich Lakenfields|       7.93434193623726| 194.2953

Sandy Roadside looks to have a substantial number of missing values.

The particulate reading average for the available data is higher than the other sites and for the other Urban Traffic sites, so imputing data to fill the missing values would likely skew the overall dataset.  In this case, it is preferable to remove the sandy roadside data from the overall dataset.

This will remove one of the Urban Traffic site types, but sufficient data will be retained (~17,000 records).

The missing hours are due to midnight in the data being represented as 24:00:00, which has not translated to a hour properly - these will be replaced with 0 in-line with standard time formatting.

The other missing values appear to be fairly evenly distributed across sites, these will be handled by imputting the mean value for each site. As the descriptive statistics suggest that the values are reasonably evenly grouped around the mean this should minimise skew while maximising retained data points.

We will also remove columns that are less relevent to the project (Time / Longitude / Latitude / Zone / LocalAuthority and Status columns)

In [769]:
#Create subset of relevent data, ignoring columns to be removed
df_tidy=df.select('Date','SiteName','SiteType','ParticulateReading','WindSpeed','WindDirection','Hour')
#Check that new dataframe fits expectations
df_tidy.show(5)

+----------+--------------------+----------------+------------------+---------+-------------+----+
|      Date|            SiteName|        SiteType|ParticulateReading|WindSpeed|WindDirection|Hour|
+----------+--------------------+----------------+------------------+---------+-------------+----+
|2025-01-01|Peterborough Gart...|Urban Background|            12.618|     13.7|        236.5|   1|
|2025-01-01|Peterborough Gart...|Urban Background|             3.325|     14.2|        236.0|   2|
|2025-01-01|Peterborough Gart...|Urban Background|             2.453|     14.2|        235.9|   3|
|2025-01-01|Peterborough Gart...|Urban Background|             1.934|     12.7|        235.9|   4|
|2025-01-01|Peterborough Gart...|Urban Background|             1.722|     12.6|        234.9|   5|
+----------+--------------------+----------------+------------------+---------+-------------+----+
only showing top 5 rows


In [770]:
#remove rows relating to Sandy Roadside and null
df_tidy=df_tidy.dropna(subset=['SiteName'])
df_tidy=df_tidy.filter(df_tidy.SiteName!='Sandy Roadside')
#Check output is as expected
df_tidy.groupBy('SiteName').count().show()

+--------------------+-----+
|            SiteName|count|
+--------------------+-----+
|Borehamwood Meado...| 8760|
|Stanford-le-Hope ...| 8760|
|Peterborough Gart...| 8760|
| Norwich Lakenfields| 8760|
|     Southend-on-Sea| 8760|
|          Wicken Fen| 8760|
|            Thurrock| 8760|
|              Sibton| 8760|
| Luton A505 Roadside| 8760|
|           Weybourne| 8760|
|            St Osyth| 8760|
+--------------------+-----+



In [771]:
#generate baseline before fixing hours value
df_tidy.select('Hour').groupBy('Hour').count().show(24)

+----+-----+
|Hour|count|
+----+-----+
|  12| 4015|
|  22| 4015|
|NULL| 4015|
|   1| 4015|
|  13| 4015|
|   6| 4015|
|  16| 4015|
|   3| 4015|
|  20| 4015|
|   5| 4015|
|  19| 4015|
|  15| 4015|
|   9| 4015|
|  17| 4015|
|   4| 4015|
|   8| 4015|
|  23| 4015|
|   7| 4015|
|  10| 4015|
|  21| 4015|
|  11| 4015|
|  14| 4015|
|   2| 4015|
|  18| 4015|
+----+-----+



In [772]:
#replace null hour values with 0
df_tidy=df_tidy.na.fill({'Hour':0})
#check changes in place
df_tidy.select('Hour').groupBy('Hour').count().show(24)


+----+-----+
|Hour|count|
+----+-----+
|  12| 4015|
|  22| 4015|
|   1| 4015|
|  13| 4015|
|   6| 4015|
|  16| 4015|
|   3| 4015|
|  20| 4015|
|   5| 4015|
|  19| 4015|
|  15| 4015|
|   9| 4015|
|  17| 4015|
|   4| 4015|
|   8| 4015|
|  23| 4015|
|   7| 4015|
|  10| 4015|
|  21| 4015|
|  11| 4015|
|  14| 4015|
|   2| 4015|
|   0| 4015|
|  18| 4015|
+----+-----+



In [773]:
#replace null ParticulateReading with site average for each value
#create a sub-table of average values by site name
#define as a function
from pyspark.sql import Column as pc

cols = [df_tidy.ParticulateReading,df_tidy.WindSpeed,df_tidy.WindDirection]

for col in cols:
  PrAvg = df_tidy.select(df_tidy.SiteName,col).groupby('SiteName').avg()
  PrAvg = PrAvg.withColumnRenamed(PrAvg.columns[1],'SiteAvg')
  #PrAvg.show()
  #Use join to add the average value to the main table
  df_tidy=df_tidy.join(PrAvg,on='SiteName',how='left')
  #Use when statement to replace null values
  df_tidy=df_tidy.withColumn('imputed', sf.ifnull(col,df_tidy.SiteAvg))
  #remove unwanted columns and rename imputed as ParticulateReading
  colName = str(col)[8:-2]
  df_tidy = df_tidy.drop('SiteAvg',colName)
  df_tidy=df_tidy.withColumnRenamed('imputed',colName)

df_tidy.show(5)



+--------------------+----------+----------------+----+------------------+------------------+------------------+
|            SiteName|      Date|        SiteType|Hour|ParticulateReading|         WindSpeed|     WindDirection|
+--------------------+----------+----------------+----+------------------+------------------+------------------+
|Peterborough Gart...|2025-01-01|Urban Background|   1|            12.618|13.699999809265137|             236.5|
|Peterborough Gart...|2025-01-01|Urban Background|   2|             3.325|14.199999809265137|             236.0|
|Peterborough Gart...|2025-01-01|Urban Background|   3|             2.453|14.199999809265137|235.89999389648438|
|Peterborough Gart...|2025-01-01|Urban Background|   4|             1.934|12.699999809265137|235.89999389648438|
|Peterborough Gart...|2025-01-01|Urban Background|   5|             1.722|12.600000381469727|234.89999389648438|
+--------------------+----------+----------------+----+------------------+------------------+---

In [774]:
df_tidy.select('SiteName','WindDirection').groupBy('SiteName').count().show()

+--------------------+-----+
|            SiteName|count|
+--------------------+-----+
|Borehamwood Meado...| 8760|
|Stanford-le-Hope ...| 8760|
|Peterborough Gart...| 8760|
| Norwich Lakenfields| 8760|
|     Southend-on-Sea| 8760|
|          Wicken Fen| 8760|
|            Thurrock| 8760|
|              Sibton| 8760|
| Luton A505 Roadside| 8760|
|           Weybourne| 8760|
|            St Osyth| 8760|
+--------------------+-----+



In [775]:
chck = ps.DataFrame(df_tidy)
chck.groupby('SiteName').count().sort_index()

,Date,SiteType,Hour,ParticulateReading,WindSpeed,WindDirection
SiteName,,,,,,
Borehamwood Meadow Park,8760,8760,8760,8760,8760,8760
Luton A505 Roadside,8760,8760,8760,8760,8760,8760
Norwich Lakenfields,8760,8760,8760,8760,8760,8760
Peterborough Garton End,8760,8760,8760,8760,8760,8760
Sibton,8760,8760,8760,8760,8760,8760
Southend-on-Sea,8760,8760,8760,8760,8760,8760
St Osyth,8760,8760,8760,8760,8760,8760
Stanford-le-Hope Roadside,8760,8760,8760,8760,8760,8760
Thurrock,8760,8760,8760,8760,8760,8760


# Create test and train datasets, ensuring that splits are suitably stratified

# Explore training dataset

# Explore models for predicting air quality levels

# Select the best model and use cross-validation to tune the hyperparameters

# Create pipeline to prepare data and run selected model on it

#Run pipeline on Test data